# modeling_v13 — Feature Diet (4-stage) · `feature_diet`

**입력**: `data/v13_fdc_pool_wf_oof.csv.gz` (11,939 WF × 655 피처, 27센서)

**제거 순서**: ① Variance Threshold(상수 제거) → ② 결측률 필터 → ③ Correlation → ④ VIF

**Floor 규칙 (필수)**: `C17`(온도), `C11`(플라즈마), `C31`(RF), `C15`·`C16`(가스)
각 센서는 **최소 1개 피처가 살아남아야 한다.**
→ 센서별 **champion 1개**를 미리 선정해 4단계 전체에서 **제거 면제**로 고정한다.
champion = (결측률 최소) → (|타깃 상관| 최대) 순, 단 **비상수(nunique>1)** 중에서만 선정.

2개 프리셋(Conservative/Balanced)을 한 번에 돌려 **비교표**를 만든다.
후속 성능 비교 대상으로 이 둘만 유지 (Aggressive 제외).

In [1]:
import time, re, json
import numpy as np, pandas as pd

DATA = "data/v13_fdc_pool_wf_oof.csv.gz"   # parquet 있으면 pd.read_parquet 로 교체 가능
df = pd.read_csv(DATA)

META, TARGET = ["C64", "fold_kf5", "C20"], "C65"
FEATS = [c for c in df.columns if c not in META + [TARGET]]
PROTECTED = ["C17", "C11", "C31", "C15", "C16"]   # 온도/플라즈마/RF/가스·가스

def sensor_of(col):
    return re.match(r"(C\d+)_", col).group(1)

print(f"WF={len(df)}, feats={len(FEATS)}, sensors={len(set(map(sensor_of, FEATS)))}")

WF=11939, feats=655, sensors=27


## 프리셋 정의

| 프리셋 | 결측률 컷 | \|상관\| 컷 | VIF 컷 |
|---|---|---|---|
| Conservative | > 0.70 | > 0.97 | > 10 |
| Balanced | > 0.50 | > 0.95 | > 10 |

`nzv`(min-max 정규화 분산) 컷은 상수 제거의 **안전망**이다. 실제로는 exact 상수(`nunique≤1`)가
먼저 걸리므로 이 데이터에선 추가 제거가 0이지만, 다른 데이터에 재사용할 때를 위해 남겨둔다.

In [2]:
PRESETS = {
    "Conservative": dict(miss=0.70, corr=0.97, vif=10.0, nzv=1e-5),
    "Balanced":     dict(miss=0.50, corr=0.95, vif=10.0, nzv=1e-5),
}

## 사전 통계 (결측률 · 유니크 수 · 정규화 분산 · 타깃 상관)

In [3]:
X = df[FEATS].astype("float32")
y = df[TARGET].astype("float64")

miss = X.isna().mean()                       # 결측률
nun  = X.nunique(dropna=True)                # 유니크 값 수
rng  = (X.max() - X.min()).replace(0, np.nan)
Xn   = (X - X.min()) / rng                   # min-max 정규화
nzv  = Xn.var(ddof=0)                         # scale-invariant near-constant 지표
tcorr = X.apply(lambda s: s.corr(y)).abs().fillna(0.0)   # |타깃 상관| (pairwise complete)

print("exact 상수(nunique<=1):", int((nun <= 1).sum()))

exact 상수(nunique<=1): 220


C:\Users\Dell3571\Documents\defect_test_prediction_MK\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\Dell3571\Documents\defect_test_prediction_MK\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## champion 선정 (센서별 floor 보장)

각 필수 센서에서 비상수 피처 중 **결측률 최소 → |타깃 상관| 최대** 로 1개를 골라
4단계 전 구간에서 제거 면제로 고정한다.

In [4]:
def pick_champions(cand):
    champs = {}
    for s in PROTECTED:
        pool = [c for c in cand if sensor_of(c) == s and nun[c] > 1]
        if not pool:                                   # 안전망(전부 상수인 극단 케이스)
            pool = [c for c in cand if sensor_of(c) == s]
        pool.sort(key=lambda c: (miss[c], -tcorr[c], c))
        champs[s] = pool[0]
    return set(champs.values()), champs

## 단계 ③ Correlation — 타깃 인지 greedy

champion을 먼저 keep 리스트에 넣고, 나머지는 **|타깃 상관| 내림차순**으로 순회.
이미 keep된 피처와 `|corr| > thr` 이면 드롭(= 타깃과 더 관련 큰 쪽이 생존). champion은 절대 드롭 안 함.

In [5]:
def corr_filter(cols, champs_set, thr):
    order = list(champs_set) + sorted([c for c in cols if c not in champs_set],
                                       key=lambda c: -tcorr[c])
    cm = X[order].corr().abs().values        # pairwise-complete Pearson
    idx = {c: i for i, c in enumerate(order)}
    kept = []
    for c in order:
        i = idx[c]
        if c in champs_set:
            kept.append(c); continue
        if any((not np.isnan(cm[i, idx[k]])) and cm[i, idx[k]] > thr for k in kept):
            continue
        kept.append(c)
    return kept

## 단계 ④ VIF — 역상관행렬 방식(반복 제거)

`VIF_i = diag(inv(corr))_i`. 중앙값 대치 + 표준화 후 상관행렬을 만들고, 임계 초과 중
**최댓값 non-champion**을 하나씩 제거하며 재계산. champion은 제거하지 않는다.

In [6]:
def vif_filter(cols, champs_set, thr):
    sub = X[cols].copy().fillna(X[cols].median())
    std = sub.std(ddof=0)
    sub = sub.loc[:, std > 0]                        # 상수화된 열 제거(안전)
    sub = (sub - sub.mean()) / sub.std(ddof=0)
    cur = list(sub.columns); Xm = sub.values
    while True:
        R = np.corrcoef(Xm, rowvar=False) + 1e-6 * np.eye(Xm.shape[1])
        vif = np.diag(np.linalg.pinv(R))
        cand = sorted([(v, i) for i, (c, v) in enumerate(zip(cur, vif))
                       if c not in champs_set and v > thr], reverse=True)
        if not cand:
            break
        drop_i = cand[0][1]
        keep = [j for j in range(len(cur)) if j != drop_i]
        cur = [cur[j] for j in keep]; Xm = Xm[:, keep]
    return cur

## 파이프라인 실행 (프리셋별)

In [7]:
def run(name, p):
    champs_set, champs = pick_champions(FEATS)
    log = {"preset": name, "champions": champs}
    s1 = [c for c in FEATS if c in champs_set or (nun[c] > 1 and nzv[c] > p["nzv"])]
    log["after_variance"] = len(s1)
    s2 = [c for c in s1 if c in champs_set or miss[c] <= p["miss"]]
    log["after_missing"] = len(s2)
    s3 = corr_filter(s2, champs_set, p["corr"]); log["after_corr"] = len(s3)
    s4 = vif_filter(s3, champs_set, p["vif"]);   log["after_vif"]  = len(s4)
    per = {s: sum(1 for c in s4 if sensor_of(c) == s) for s in PROTECTED}
    log["protected_counts"] = per
    log["floor_ok"] = all(v >= 1 for v in per.values())
    log["n_sensors_final"] = len(set(map(sensor_of, s4)))
    log["final_feats"] = s4
    return log

results = {}
for name, p in PRESETS.items():
    t = time.time(); results[name] = run(name, p)
    results[name]["seconds"] = round(time.time() - t, 2)
    print(f"{name:13s} done in {results[name]['seconds']}s  floor={'OK' if results[name]['floor_ok'] else 'FAIL'}")

Conservative  done in 30.7s  floor=OK
Balanced      done in 15.34s  floor=OK


## 비교표 — 단계별 생존 피처 수

In [8]:
rows = [[n, 655, results[n]["after_variance"], results[n]["after_missing"],
         results[n]["after_corr"], results[n]["after_vif"],
         results[n]["n_sensors_final"],
         "OK" if results[n]["floor_ok"] else "FAIL", results[n]["seconds"]]
        for n in PRESETS]
summary = pd.DataFrame(rows, columns=["preset","start","①after_var","②after_miss",
                                      "③after_corr","④after_vif","sensors","floor","sec"])
summary

,preset,start,①after_var,②after_miss,③after_corr,④after_vif,sensors,floor,sec
0,Conservative,655,435,435,226,141,25,OK,30.70
1,Balanced,655,435,344,189,126,26,OK,15.34


## 필수 5센서 최종 피처 수 (모두 ≥1 이어야 함)

In [9]:
floor_tbl = pd.DataFrame({n: results[n]["protected_counts"] for n in PRESETS}).T
floor_tbl.index.name = "preset"
assert (floor_tbl >= 1).all().all(), "FLOOR VIOLATION!"
print("champions:", results["Balanced"]["champions"])
floor_tbl

champions: {'C17': 'C17_max_step4', 'C11': 'C11_min_step4', 'C31': 'C31_mean_step4', 'C15': 'C15_max_step1', 'C16': 'C16_max_step1'}


,C17,C11,C31,C15,C16
preset,,,,,
Conservative,4,12,3,3,3
Balanced,3,11,2,3,3


## 산출물 저장 (프리셋별 선택 피처 목록 + 요약)

In [10]:
summary.to_csv("feature_diet_summary.csv", index=False)
selected = {n: results[n]["final_feats"] for n in PRESETS}
with open("feature_diet_selected.json", "w", encoding="utf-8") as f:
    json.dump({"presets": PRESETS,
               "protected": PROTECTED,
               "champions": {n: results[n]["champions"] for n in PRESETS},
               "protected_counts": {n: results[n]["protected_counts"] for n in PRESETS},
               "selected": selected}, f, ensure_ascii=False, indent=2)
print("saved: feature_diet_summary.csv, feature_diet_selected.json")
for n in PRESETS:
    print(f"  {n:13s} -> {len(selected[n])} feats")

saved: feature_diet_summary.csv, feature_diet_selected.json
  Conservative  -> 141 feats
  Balanced      -> 126 feats


---
### 다음 단계 제안
선택된 셋을 baseline(COMBINED_FULL, valid 48.76)의 **시간/레짐 7피처와 결합**해
LGBM 5-fold OOF(및 `GroupKFold(C20)` 정직-CV)로 각 프리셋 RMSE를 비교하면,
"피처 수 vs 성능" 트레이드오프에서 최적 프리셋을 고를 수 있다.